# Demo: cortical microdomain self-organisation

This is a complete training-and-presentation workflow for a model of salt-and-pepper cortical maps (micr-GCAL). Running all cells trains the model for 2 epochs only when a completed archive is absent. The notebook contains configuration and helper calls only; collection, checkpointing, measurements, plotting, animation, synthetic probes, and scatter permutation live in `helpers/microdomain_demo.py`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'demo_microdomains' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from demo_microdomains.helpers.microdomain_demo import (
    animation_html,
    animate_dimensionality_learning,
    animate_map_learning,
    animate_rotating_umap_grid,
    animate_scattered_map_learning,
    animate_synthetic_learning,
    animate_weight_learning,
    animate_robustness_learning,
    ensure_github_assets,
    estimate_demo_archive_gib,
    github_demo_config,
    macaque_displacement_table,
    plot_lgn_inputs_and_statistics,
    plot_macaque_displacement_links,
    plot_macaque_displacement_summary,
    run_macaque_displacement_demo,
    train_or_load_demo,
)

## 0. Train or load the archived run

The configuration uses a 100×100 L4 microdomain sheet, 80×80 natural-image crops, and two shuffled epochs. `train_or_load_demo` protects a completed archive: it loads existing data immediately and launches the expensive collection only when the archive is missing or incomplete.

In [ ]:
config = github_demo_config()
estimate_demo_archive_gib(config)

In [ ]:
archive = train_or_load_demo(config)
archive.manifest['status'], archive.n_frames

In [ ]:
assets = ensure_github_assets(archive)
assets

## 1. Fixed LGN drive

The held-out crops provide the common reference for intensity, exact-zero sparsity, spatial power, and 95%-variance input dimensionality.

<img src="demo_assets/microdomain/lgn_inputs.png" alt="LGN inputs and statistics" width="800">

In [ ]:
plot_lgn_inputs_and_statistics(archive)

## 2. Orientation-map formation

Orientation preference, horizontal retinotopy, the zoomed central quarter of the black retinotopic fishnet, and orientation Fourier power evolve together. Vertical colour scales identify each mapped quantity; dark Fourier pixels indicate power and white indicates no power.

![Orientation map formation](demo_assets/microdomain/map_learning.gif)

In [ ]:
animation_html(animate_map_learning(archive))

## 3. Afferent and lateral plasticity

Afferent fields and cross-domain excitation (CDE) use the same perceptually uniform, normalised-weight scale, with exact zero shown in white. CDE fields are selected only when their source-centred crop lies fully inside the sheet, so no border padding is displayed.

![Afferent and lateral plasticity](demo_assets/microdomain/weight_learning.gif)

In [ ]:
animation_html(animate_weight_learning(archive))

## 4. Tracked synthetic reconstruction

The synthetic face was evaluated at every learning snapshot. The row shows the fixed input, settled L4 activity, decoder reconstruction, and score without interpolating unrecorded probes. All image panels use the same white-at-zero grey map, with separate LGN and L4 activity scales.

![Tracked face reconstruction](demo_assets/microdomain/synthetic_learning.gif)

In [ ]:
animation_html(animate_synthetic_learning(archive))

## 5. PCA geometry and dimensionality

Three leading spatial PCA components evolve in separate viridis panels with one shared normalised-loading scale. The final column follows the V1/LGN dimensionality ratio.

![PCA geometry and dimensionality](demo_assets/microdomain/dimensionality.gif)

In [ ]:
animation_html(animate_dimensionality_learning(archive))

## 6. Noise robustness

The fixed LGN input is followed by matched 20×20 windows of clean and noise-intensity 0.06 activity, chosen once from an active region and held fixed through learning. The final panel follows population clean/noisy cosine stability.

![Noise robustness](demo_assets/microdomain/robustness.gif)

In [ ]:
animation_html(animate_robustness_learning(archive))

## 7. Wiring-efficiency scatter experiment

The seeded Gaussian local permutation from `wiring_efficiency.ipynb` is calibrated to a requested mean displacement in cell units. It is held fixed across learning and applied consistently to orientation preference and receptive-field centroid identity. Horizontal retinotopy precedes the zoomed central-quarter fishnet, followed by Fourier power.

![Scatter permutation](demo_assets/microdomain/scattered_learning.gif)

In [ ]:
scatter_displacement = 2.0
animation_html(
    animate_scattered_map_learning(
        archive, mean_displacement=scatter_displacement
    )
)

## 8. Cellular orientation displacement in macaque V1

This analysis uses the three densest 850×850 µm fields (`MB_1`, `MB_2`, and `MA_2`) from the public Chen et al. macaque V1 cellular-orientation dataset. Each dot is one significantly tuned cell, coloured by one of the 12 measured orientations. We smooth all measurements with one shared Gaussian and measure every neuron against the nearest location where a leave-one-cell-out smooth map predicts exactly its orientation. To prevent extreme contour matches from dominating the summary, displacements above 350 µm are excluded from the mean and from the later linked subsample.

At the default Gaussian σ of 100 µm and maximum included displacement of 350 µm, the inferred mean displacements are:

| Dataset | Included cells | Mean displacement |
|---|---:|---:|
| `MB_1` | 792 | 91.4 µm |
| `MB_2` | 729 | 82.5 µm |
| `MA_2` | 395 | 92.3 µm |

<img src="demo_assets/microdomain/macaque_displacement_summary.png" alt="Macaque cellular scatter and smooth maps" width="1050">

In [ ]:
macaque_smoothing_sigma_um = 100.0
macaque_max_displacement_um = 350.0
macaque_results = run_macaque_displacement_demo(
    smoothing_sigma_um=macaque_smoothing_sigma_um,
    max_displacement_um=macaque_max_displacement_um,
)
plot_macaque_displacement_summary(macaque_results)
macaque_displacement_table(macaque_results)

### Example cell-to-map correspondences

To make the displacement concrete, a reproducible sparse subset of cells is overlaid on a faint version of each smooth map. Every solid black segment starts at a measured soma and ends at the nearest exact same-orientation location predicted by the smooth map rebuilt without that cell; the black × marks the predicted endpoint. Only a subset is drawn to keep the links legible, while the title retains the displacement averaged over every tuned cell.

<img src="demo_assets/microdomain/macaque_displacement_links.png" alt="Sparse macaque displacement links" width="1050">

In [ ]:
macaque_link_examples = 20
plot_macaque_displacement_links(
    macaque_results, n_links=macaque_link_examples, map_alpha=0.28
)

## 9. Rotating 3D response geometry

This reproduces the four-way 3D UMAP comparison in the paper. From left to right, the panels embed raw static-grating pixels, topographic-model responses, salt-and-pepper-model responses, and high-arousal responses from Stringer et al. (2021) random-phase recording 1 only. Each point is coloured by grating orientation modulo 180°. The four cameras rotate together around the vertical UMAP axis, making the full geometry visible.

<img src="demo_assets/microdomain/rotating_umap.gif" alt="Rotating four-panel 3D UMAP comparison" width="1050">

In [ ]:
umap_rotation_frames = 48
umap_animation_asset = assets['rotating_umap']